# 给片段补充章节标题

片段离开原文后，正文里未必还保留书名和章节名。CCH（Contextual Chunk Headers）会在建立索引时，把片段所属的标题放到正文前面，再一起用于检索。标题只帮助查找，回答时仍应引用原文。

下面给《南瓜书》的每一页补上真实章名，再用同一套 BM25 和相同返回数量比较。代码先完成两次检索，再读取正确页码做检查。

In [1]:
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

import json

from common.dataset import chapter_title
from common.eval_utils import emit_tutorial_audit
from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation

case_rows = load_query_catalog()
queries = {row['id']: row['query'] for row in case_rows}
pages = load_pdf_pages()
plain_search = build_bm25_search(pages)
pages_with_titles = [
    {'page': row['page'], 'text': f"《南瓜书》 {chapter_title(row['page'])} {row['text']}"}
    for row in pages
]
title_search = build_bm25_search(pages_with_titles)

def run(case_id):
    question = queries[case_id]
    before = plain_search(question, top_k=8)
    after = title_search(question, top_k=8)
    return question, before, after

main_question, main_before, main_after = run('model_selection_with_intro_scope')
check_question, check_before, check_after = run('random_forest_feature_sampling')

# 检索结束后再读取评估页码。
annotations = {
    case_id: load_annotation(case_id)
    for case_id in ('model_selection_with_intro_scope', 'random_forest_feature_sampling')
}

def rank(results, expected_pages):
    expected = set(int(page) for page in expected_pages)
    return next((index for index, item in enumerate(results, 1) if item.page in expected), None)

def result_metrics(results, expected_pages):
    pages = [int(item.page) for item in results]
    expected = set(int(page) for page in expected_pages)
    found = expected.intersection(pages)
    return {
        'pages': pages,
        'first_required_rank': next((index for index, page in enumerate(pages, 1) if page in expected), None),
        'required_page_coverage': len(found) / len(expected) if expected else 0.0,
    }

main_expected = annotations['model_selection_with_intro_scope']['expected_pages']
main_before_rank = rank(main_before, main_expected)
main_after_rank = rank(main_after, main_expected)
print('主要问题：', main_question)
print('不加标题：', [item.page for item in main_before], '；必要页：', main_before_rank or '前 8 条未出现')
print('补上章名：', [item.page for item in main_after], '；必要页排名：', main_after_rank)
emit_tutorial_audit({
    'case_id': 'model_selection_with_intro_scope',
    'method': '给片段补充章节标题（CCH）',
    'role': 'main',
    'before': result_metrics(main_before, main_expected),
    'after': result_metrics(main_after, main_expected),
})
assert main_before_rank is None and main_after_rank == 2

check_expected = annotations['random_forest_feature_sampling']['expected_pages']
check_before_rank = rank(check_before, check_expected)
check_after_rank = rank(check_after, check_expected)
print('\n复查问题：', check_question)
print('不加标题：', [item.page for item in check_before], '；必要页排名：', check_before_rank)
print('补上章名：', [item.page for item in check_after], '；必要页排名：', check_after_rank)
emit_tutorial_audit({
    'case_id': 'random_forest_feature_sampling',
    'method': '给片段补充章节标题（CCH）',
    'role': 'check',
    'before': result_metrics(check_before, check_expected),
    'after': result_metrics(check_after, check_expected),
    'check_purpose': '确认没有改坏',
})
assert check_before_rank == 1 and check_after_rank == 1


主要问题： 《南瓜书》绪论里说，机器学习算法之间有没有绝对更好的一个？
不加标题： [2, 14, 15, 18, 31, 149, 88, 101] ；必要页： 前 8 条未出现
补上章名： [15, 17, 16, 14, 140, 149, 18, 88] ；必要页排名： 2



复查问题： 随机森林相对 Bagging 做了哪项随机化扩展？
不加标题： [100, 104, 7, 109, 149, 29, 113, 23] ；必要页排名： 1
补上章名： [100, 104, 7, 109, 29, 149, 113, 23] ；必要页排名： 1


主要问题明确问《南瓜书》绪论，但第 17 页的正文没有重复书名和章名。不加标题时，第 17 页没有进入前 8 条；补上真实章名后，它排到第 2。随机森林复查题在改动前后都把第 100 页排在第 1。

标题必须来自可靠的文档结构。标题写错、过于宽泛，或片段同时属于多个主题时，补标题也可能把检索带偏。

## CCH 的原理和适用范围

CCH（Contextual Chunk Headers，给片段补章节标题）是在建立索引时把片段所属的文档或章节标题放到正文前，再对“标题 + 原片段”编码。片段单独看时可能只写“该方法”“这个公式”，或者省略书名。补上标题后，问题中的书名和章节名更容易匹配。示意图如下：

![CCH 将标题附加到片段](figures/cch.png)

实现通常分三步：从可靠的文档结构读取标题（或让模型根据全文生成文档标题）；把标题与原片段拼接，仅用于建立检索字段；命中后仍返回原片段和原 metadata。标题应来自目录、标题层级或人工核对，不能让模型凭空改写资料主题。标题太宽泛、对应关系错误或一个片段跨越多个章节时，加入 CCH 反而会把检索带偏。

CCH 和 Contextual Retrieval（给每个片段生成所属背景）的区别在于：CCH 通常使用文档或章节标题，多个片段可以共享同一个前缀；后者为每个片段单独生成一两句背景，用来区分同一文档中的不同位置。标题成本较低，可以先试；逐片段生成背景需要更多模型调用。


## 把可靠章节标题写入检索字段

本页正式实验已经演示了标题增强；下面只说明检索字段保留标题、回答字段保留原文的代码边界，不产生另一份结果。

```python
def attach_reliable_header(chunk: dict, title: str) -> dict:
    # 检索字段加标题，回答字段保留原文。
    original = chunk["text"]
    header = title.strip()
    return {
        **chunk,
        "search_text": f"{header}\n{original}",
        "answer_text": original,
        "metadata": {**chunk.get("metadata", {}), "chapter_title": header},
    }

# title 必须来自目录、页面范围或人工核对的结构，不能从 expected_pages 反推。
```
